# iDeepLBCE stratified k-fold cross-validation

Set `DATA_ROOT` to the parent of `train_ibce`, select `FEATURES`, `MODELS`, and `K`, then run all cells. Only training-partition CSVs are loaded; internal and external test datasets remain outside this workflow.

CSV loading, feature dimensions/group concatenation, classical model settings, and JSON-based deep-learning construction are preserved. CSVs are headerless: first column is label (0 = non-epitope, 1 = epitope); remaining columns are numeric features. Each filename begins with the feature prefix, with one match per prefix. Each folder's FASTA and feature files preserve the same sample order. F1 combines AAC/DPC/PAAC/APAAC/PCP; F2 combines ONEHOT/PSSM/PSTPP. Add custom combinations to `GROUPS`.

Every selected model/feature uses the same shuffled stratified outer folds. ML fits a new scaler/model on each outer training fold. DL splits that training fold again for early stopping: scaler and class weights use only the inner fitting subset, and outer test-fold labels never control stopping. DL therefore fits fewer samples per fold than ML; this preserves validation-based stopping without using outer evaluation data. Each DL fold starts with fresh weights.

Metrics: ACC, ROC_AUC, AP (average precision), MCC, F1, Sensitivity, Specificity. Threshold defaults to 0.5 and is fixed before evaluation. Summaries report unweighted fold means and sample SD (`ddof=1`), not standard errors or repeated-seed variability. Pooled out-of-fold scores are saved separately and can differ from fold means.

Precomputed label-derived PSTPP lookup tables must also respect fold boundaries for a fully leakage-controlled CV estimate. This notebook preserves your precomputed CSV workflow and cannot reconstruct fold-specific features from the CSV alone. Sequence/antigen grouping is not enforced by stratified sample folds.

Requires pandas, scikit-learn, and joblib; TensorFlow/Keras 3 only for DL, XGBoost only if selected. Mount Drive first in Colab; enable a GPU for DL. No hyperparameter search or independent test evaluation is performed.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from dataclasses import dataclass
import copy
import csv
import json
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score, matthews_corrcoef,
    roc_auc_score,
)
import joblib
import time


## Settings

Select a single feature/model as a string, or multiple selections as lists. `K = 5` gives five-fold CV. For custom concatenations, define a name in `GROUPS` and select that name. `CV_SEED` fixes fold membership; `MODEL_SEED` fixes model initialization.


In [ ]:
DATA_ROOT = Path('features')
DATA_DIRS = {'train_ibce': DATA_ROOT / 'train_ibce'}
CSV_OVERRIDES = {}  # Example: {('train_ibce', 'PSTPP'): Path('/path/PSTPP.csv')}
MODEL_JSON_DIR = None  # Optional directory containing the original JSON model files
OUTPUT_ROOT = Path('.') / '_cv_results_'

FEATURES = ['PSTPP']  # 'F1', 'F2', 'ONEHOT', or ['PSTPP', 'F1', 'F2']
MODELS = ['CNN']      # 'RF' or ['LR', 'RF', 'ET', 'XGB', 'CNN', 'FCNN', 'RNN', 'LSTM', 'GRU']
K = 2
CV_SEED = 42
MODEL_SEED = 42
THRESHOLD = 0.50
VAL_FRACTION = 0.20  # Inner validation fraction of each DL outer-training fold
MAX_EPOCHS = 200
BATCH_SIZE = 32
PATIENCE = 50
MONITOR = 'val_loss'
MODE = 'min'
N_JOBS = 2
SAVE_MODELS = True
EXPECTED_ROWS = dict(train_ibce=9925)  # Use {} for intentionally different datasets

FEATURES = [FEATURES] if isinstance(FEATURES, str) else list(FEATURES)
MODELS = [MODELS] if isinstance(MODELS, str) else list(MODELS)
FEATURES = [re.sub(r'[^A-Z0-9]', '', f.upper()) for f in FEATURES]
MODELS = [m.upper() for m in MODELS]
ML_NAMES = {'LR', 'RF', 'ET', 'XGB'}
DL_NAMES = {'CNN', 'FCNN', 'RNN', 'LSTM', 'GRU'}
if not FEATURES or not MODELS or len(FEATURES)!=len(set(FEATURES)) or len(MODELS)!=len(set(MODELS)):
    raise ValueError('Select nonempty, unique features and models.')
if set(MODELS)-(ML_NAMES|DL_NAMES): raise ValueError('Unknown model selected.')
if isinstance(K, bool) or not isinstance(K, int) or K<2: raise ValueError('K must be an integer >= 2.')
if not 0<THRESHOLD<1 or not 0<VAL_FRACTION<1: raise ValueError('Invalid threshold or inner validation fraction.')
ML_JOBS = {f:[m for m in MODELS if m in ML_NAMES] for f in FEATURES}
DL_JOBS = {f:[m for m in MODELS if m in DL_NAMES] for f in FEATURES}
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR/'run_settings.json').write_text(json.dumps(dict(
    data_root=str(DATA_ROOT), features=FEATURES, models=MODELS, k=K,
    cv_seed=CV_SEED, model_seed=MODEL_SEED, threshold=THRESHOLD,
    inner_validation_fraction=VAL_FRACTION, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, patience=PATIENCE, monitor=MONITOR, mode=MODE,
    save_models=SAVE_MODELS, sd_ddof=1),indent=2))
print('Results directory:', RUN_DIR)


Results directory: _cv_results_\20261122T114501_111156Z


In [3]:
DIMENSIONS = dict(AAC=20, DPC=400, PAAC=25, APAAC=30, PCP=9, PSSM=62, PSTPP=23, ONEHOT=525)
GROUPS = {'F1': ('AAC','DPC','PAAC','APAAC','PCP'), 'F2': ('ONEHOT','PSSM','PSTPP')}
PREFIXES = {'ONEHOT': ('onehot','one-hot','one_hot','one hot','ohe')}
ID_COLUMNS = ('peptide_id','seq_id','sample_id','orig_idx','sequence')
TESTS = ()  # CV loads only train_ibce; independent test folders are not used.

@dataclass
class Dataset:
    X: pd.DataFrame
    y: np.ndarray


def find_csv(split, block):
    # print(f"Split: {split} block: {block}") # <-- Test
    override = CSV_OVERRIDES.get((split, block))
    if override is not None:
        return Path(override)
    folder = Path(DATA_DIRS[split])
    aliases = PREFIXES.get(block, (block.lower(),))
    # print(f"aliases: {aliases} folder: {folder}") # <-- Test
  
    candidates = sorted(p for p in folder.iterdir() if p.is_file() and
                        p.suffix.lower() == '.csv' and p.stem.lower().startswith(aliases))
    if len(candidates) != 1:
        raise ValueError(f'{split}/{block}: expected one CSV in {folder}, found {candidates}. Set CSV_OVERRIDES.')
    return candidates[0]

def read_block(path, block):

    frame = pd.read_csv(
        path,
        dtype=str,
        header=None,
        keep_default_na=False,
        encoding="utf-8-sig",
        engine="python",
    )  # pd.read_csv(path, dtype=str, keep_default_na=False)


    labels = pd.to_numeric(frame.iloc[:, 0], errors="raise")
    if set(labels.unique()) != {0, 1}:
        raise ValueError(f"{path}: column 1 must contain both binary labels, 0 and 1.")

    X = (
        frame.iloc[:, 1:]
        .apply(pd.to_numeric, errors="raise").astype("float32")
    )
    if X.shape[1] != DIMENSIONS[block] or not np.isfinite(X.to_numpy()).all():
        raise ValueError(
            f"{path}: expected {DIMENSIONS[block]} finite numeric features, got {X.shape[1]} columns."
        )
    return Dataset(X.add_prefix(f"{block}:"), labels.to_numpy(dtype=int))  # , ids)


def load_feature(feature_name):
    feature = re.sub(r"[^A-Z0-9]", "", feature_name.upper())
    blocks = GROUPS.get(feature, (feature,))
    # print(blocks) # <--
    if any(block not in DIMENSIONS for block in blocks):
        raise ValueError(
            f"Unknown feature {feature_name}. Choose from {list(DIMENSIONS) + list(GROUPS)}."
        )
    result = {}
    for split in ("train_ibce", *TESTS):
        datasets = [read_block(find_csv(split, block), block) for block in blocks]
        base = datasets[0]
        # ids = base.ids.copy()
        expected = EXPECTED_ROWS.get(split)
        if expected is not None and len(base.y) != expected:
            raise ValueError(
                f"{split}: found {len(base.y)} peptides, expected {expected}."
            )
        
        for other in datasets[1:]:
            if not np.array_equal(base.y, other.y):
                raise ValueError(
                    f"{split}/{feature}: feature blocks differ in row count or label order."
                )
            
        X = pd.concat([d.X for d in datasets], axis=1)
        if split != "train_ibce" and set(X.columns) != set(
            result["train_ibce"].X.columns
        ):
            raise ValueError(
                f"{split}/{feature}: feature headers differ from training."
            )
        if split != "train_ibce":
            X = X.loc[:, result["train_ibce"].X.columns]
        result[split] = Dataset(X, base.y)
        print(
            f"{split}/{feature}: {X.shape}, class counts {np.bincount(base.y).tolist()}"
        )
    return result

## Shared fold evaluation and exports

Rows retain zero-based original CSV indices. Fold metrics are saved after every completed fold. Shared fold assignments permit paired comparison across selected models/features. OOF means out-of-fold.


In [4]:
METRICS = ['ACC', 'ROC_AUC', 'AP', 'MCC', 'F1', 'Sensitivity', 'Specificity']
FOLD_CACHE = {}
LABEL_REFERENCE = None
ALL_ROWS = []

def folds_for(train):
    global LABEL_REFERENCE
    if LABEL_REFERENCE is None:
        LABEL_REFERENCE = train.y.copy()
    elif not np.array_equal(LABEL_REFERENCE, train.y):
        raise ValueError('Selected feature sets have different row counts/label order.')
    if not FOLD_CACHE:
        if np.bincount(train.y, minlength=2).min() < K:
            raise ValueError('Each class must contain at least K samples.')
        splitter = StratifiedKFold(n_splits=K, shuffle=True, random_state=CV_SEED)
        FOLD_CACHE['splits'] = list(splitter.split(train.X, train.y))
        assignments = np.zeros(len(train.y), dtype=int)
        for fold, (_, test_ids) in enumerate(FOLD_CACHE['splits'],1): assignments[test_ids] = fold
        pd.DataFrame(dict(row_index=np.arange(len(train.y)),y_true=train.y,fold=assignments)).to_csv(RUN_DIR/'fold_assignments.csv',index=False)
    return FOLD_CACHE['splits']

def metric_values(y, p):
    p=np.asarray(p,dtype=float).reshape(-1)
    if p.shape!=y.shape or not np.isfinite(p).all() or np.any((p<0)|(p>1)):
        raise ValueError('Invalid positive-class probabilities.')
    pred=(p>=THRESHOLD).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return dict(n=len(y),ACC=float(np.mean(y==pred)),ROC_AUC=roc_auc_score(y,p),
        AP=average_precision_score(y,p),MCC=matthews_corrcoef(y,pred),F1=f1_score(y,pred,zero_division=0),
        Sensitivity=tp/(tp+fn),Specificity=tn/(tn+fp),TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp))

def evaluate_fold(train, test_ids, probabilities, model, feature, fold, seed):
    p=np.asarray(probabilities,dtype=float).reshape(-1); y=train.y[test_ids]
    values=metric_values(y,p)
    if np.unique(p>=THRESHOLD).size==1 or values['ROC_AUC']<0.5:
        warnings.warn(f'{model}/{feature}/fold {fold}: inspect score distributions and alignment.')
    meta=dict(dataset='train_ibce_cv',model=model,feature_set=feature,fold=fold,seed=seed,threshold=THRESHOLD)
    pd.DataFrame(dict(**meta,row_index=test_ids,y_true=y,proba=p,y_pred=(p>=THRESHOLD).astype(int))).to_csv(
        RUN_DIR/f'{model}_{feature}_fold{fold}_predictions.csv',index=False)
    return dict(**meta,**values)

def save_metrics(rows,prefix):
    if not rows: return
    frame=pd.DataFrame(rows).sort_values(['feature_set','model','fold'])
    frame.to_csv(RUN_DIR/f'{prefix}_metrics_per_fold.csv',index=False)
    summary=frame.groupby(['dataset','model','feature_set'])[METRICS].agg(['count','mean','std'])
    summary.columns=['_'.join(c) for c in summary.columns]
    summary=summary.reset_index(); summary.to_csv(RUN_DIR/f'{prefix}_metrics_summary.csv',index=False)
    formatted=summary[['dataset','model','feature_set']].copy()
    formatted['folds']=summary[f'{METRICS[0]}_count']
    for m in METRICS:
        formatted[m]=[f'{mean:.4f} ± {sd:.4f}' if pd.notna(sd) else f'{mean:.4f} ± undefined' for mean,sd in zip(summary[m+'_mean'],summary[m+'_std'])]
    formatted.to_csv(RUN_DIR/f'{prefix}_mean_std.csv',index=False)
    print(f'\n{prefix}: mean ± sample SD across completed folds')
    print(formatted.to_string(index=False))
    return formatted

def save_oof(train,probabilities,model,feature):
    if not np.isfinite(probabilities).all(): raise ValueError('Incomplete OOF predictions.')
    assignments=np.zeros(len(train.y),dtype=int)
    for fold,(_,ids) in enumerate(folds_for(train),1): assignments[ids]=fold
    pd.DataFrame(dict(row_index=np.arange(len(train.y)),fold=assignments,y_true=train.y,
        proba=probabilities,y_pred=(probabilities>=THRESHOLD).astype(int))).to_csv(RUN_DIR/f'{model}_{feature}_oof_predictions.csv',index=False)
    pd.DataFrame([dict(model=model,feature_set=feature,**metric_values(train.y,probabilities))]).to_csv(RUN_DIR/f'{model}_{feature}_pooled_oof_metrics.csv',index=False)


## Classical classifiers

The original classifier constructors and parameters are retained. Individual features use the original F2 fallback settings (the previous explanatory text incorrectly said F1). Every fold fits a fresh scaler and classifier on the outer training subset.


In [5]:
ML_PARAMETERS = {'F2': {'LR': {'solver': 'lbfgs', 'penalty': 'l2', 'C': 1},
        'RF': {'n_estimators': 200,
               'min_samples_split': 5,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'ET': {'n_estimators': 500,
               'min_samples_split': 5,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'XGB': {'subsample': 0.8,
                'n_estimators': 500,
                'max_depth': 10,
                'learning_rate': 0.01,
                'colsample_bytree': 0.8}},
 'F1': {'LR': {'solver': 'liblinear', 'penalty': 'l2', 'C': 0.1},
        'RF': {'n_estimators': 500,
               'min_samples_split': 10,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'ET': {'n_estimators': 500,
               'min_samples_split': 2,
               'min_samples_leaf': 2,
               'max_features': 'sqrt',
               'max_depth': 10},
        'XGB': {'subsample': 0.8,
                'n_estimators': 500,
                'max_depth': 10,
                'learning_rate': 0.01,
                'colsample_bytree': 0.8}}}

In [6]:
def fit_classical(name, feature, train, train_ids, test_ids, fold, seed):
    from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
    from sklearn.linear_model import LogisticRegression

    factory = {'LR': LogisticRegression, 'RF': RandomForestClassifier, 'ET': ExtraTreesClassifier}
    if name == 'XGB':
        from xgboost import XGBClassifier
        factory['XGB'] = XGBClassifier
    source = feature if feature in ML_PARAMETERS else 'F2'
    settings = dict(ML_PARAMETERS[source][name], random_state=seed)
    if name != 'LR':
        settings['n_jobs'] = N_JOBS
    if name == 'XGB':
        settings.update(objective='binary:logistic', eval_metric='logloss')
    scaler = StandardScaler().fit(train.X.iloc[train_ids])
    classifier = factory[name](**settings).fit(scaler.transform(train.X.iloc[train_ids]), train.y[train_ids])
    positive_column = list(classifier.classes_).index(1)
    prefix = RUN_DIR / f'{name}_{feature}_fold{fold}'
    Path(f'{prefix}_settings.json').write_text(json.dumps(dict(parameter_source=source,
        parameters=settings,scaler='StandardScaler',fold=fold,seed=seed,n_fit=len(train_ids)),indent=2))
    if SAVE_MODELS:
        joblib.dump(dict(scaler=scaler,model=classifier,feature_columns=list(train.X.columns)),f'{prefix}_model.joblib')
    probabilities=classifier.predict_proba(scaler.transform(train.X.iloc[test_ids]))[:,positive_column]
    return probabilities

ml_rows=[]
for feature, models in ML_JOBS.items():
    if not models: continue
    train=load_feature(feature)['train_ibce']
    splits=folds_for(train)
    for model in models:
        oof=np.full(len(train.y),np.nan)
        for fold,(train_ids,test_ids) in enumerate(splits,1):
            started=time.perf_counter(); seed=MODEL_SEED+fold-1
            p=fit_classical(model,feature,train,train_ids,test_ids,fold,seed)
            oof[test_ids]=p
            row=evaluate_fold(train,test_ids,p,model,feature,fold,seed)
            row.update(n_outer_train=len(train_ids),n_fit=len(train_ids),n_inner_val=0,seconds=time.perf_counter()-started)
            ml_rows.append(row); ALL_ROWS.append(row)
            save_metrics(ml_rows,'classical'); save_metrics(ALL_ROWS,'all_models')
        save_oof(train,oof,model,feature)


train_ibce/PSTPP: (9925, 23), class counts [5485, 4440]

classical: mean ± sample SD across completed folds
      dataset model feature_set  folds                ACC            ROC_AUC                 AP                MCC                 F1        Sensitivity        Specificity
train_ibce_cv   XGB       PSTPP      1 0.9432 ± undefined 0.9915 ± undefined 0.9903 ± undefined 0.8850 ± undefined 0.9361 ± undefined 0.9302 ± undefined 0.9537 ± undefined

all_models: mean ± sample SD across completed folds
      dataset model feature_set  folds                ACC            ROC_AUC                 AP                MCC                 F1        Sensitivity        Specificity
train_ibce_cv   XGB       PSTPP      1 0.9432 ± undefined 0.9915 ± undefined 0.9903 ± undefined 0.8850 ± undefined 0.9361 ± undefined 0.9302 ± undefined 0.9537 ± undefined

classical: mean ± sample SD across completed folds
      dataset model feature_set  folds             ACC         ROC_AUC              AP             

## Deep learning

The original embedded JSON configurations and `build_deep_model` are unchanged, including input reshaping, activations, optimizer, and loss. Early stopping and learning-rate callbacks use an inner stratified validation split within each outer training fold. Fresh model weights, scalers, and class weights are calculated per fold.


In [ ]:
MODEL_FILES = {'CNN': 'model_cnn.json', 'FCNN': 'model_dnn.json', 'LSTM': 'model_lstm.json', 'GRU': 'model_gru.json', 'RNN': 'model_rnn.json'}
MODEL_CONFIGS = json.loads('{"CNN":{"config":{"name":"sequential_5","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,48,1],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_5","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"Conv1D","config":{"name":"conv1d","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"filters":64,"kernel_size":[8],"strides":[1],"padding":"same","data_format":"channels_last","dilation_rate":[1],"groups":1,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"bias_constraint":null},"registered_name":null,"build_config":{"input_shape":[null,48,1]}},{"module":"keras.layers","class_name":"MaxPooling1D","config":{"name":"max_pooling1d","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"pool_size":[2],"padding":"valid","strides":[2],"data_format":"channels_last"},"registered_name":null},{"module":"keras.layers","class_name":"BatchNormalization","config":{"name":"batch_normalization","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"axis":-1,"momentum":0.99,"epsilon":0.001,"center":true,"scale":true,"beta_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"gamma_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"moving_mean_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"moving_variance_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"beta_regularizer":null,"gamma_regularizer":null,"beta_constraint":null,"gamma_constraint":null,"synchronized":false},"registered_name":null,"build_config":{"input_shape":[null,24,64]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_6","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.4,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Conv1D","config":{"name":"conv1d_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"filters":32,"kernel_size":[8],"strides":[1],"padding":"same","data_format":"channels_last","dilation_rate":[1],"groups":1,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"bias_constraint":null},"registered_name":null,"build_config":{"input_shape":[null,24,64]}},{"module":"keras.layers","class_name":"MaxPooling1D","config":{"name":"max_pooling1d_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"pool_size":[2],"padding":"valid","strides":[2],"data_format":"channels_last"},"registered_name":null},{"module":"keras.layers","class_name":"BatchNormalization","config":{"name":"batch_normalization_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"axis":-1,"momentum":0.99,"epsilon":0.001,"center":true,"scale":true,"beta_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"gamma_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"moving_mean_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"moving_variance_initializer":{"module":"keras.initializers","class_name":"Ones","config":{},"registered_name":null},"beta_regularizer":null,"gamma_regularizer":null,"beta_constraint":null,"gamma_constraint":null,"synchronized":false},"registered_name":null,"build_config":{"input_shape":[null,12,32]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_7","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Flatten","config":{"name":"flatten_3","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"data_format":"channels_last"},"registered_name":null,"build_config":{"input_shape":[null,12,32]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_11","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":192,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,384]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_8","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_12","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,192]}}],"build_input_shape":[null,48,1]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"FCNN":{"config":{"name":"sequential","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":352,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,48]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.5,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":352,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,352]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.5,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,352]}}],"build_input_shape":[null,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"LSTM":{"config":{"name":"sequential_4","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,1,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_4","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"LSTM","config":{"name":"lstm","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":true,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":96,"activation":"relu","recurrent_activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"unit_forget_bias":true,"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.1,"recurrent_dropout":0.0,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,48]}},{"module":"keras.layers","class_name":"LSTM","config":{"name":"lstm_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":false,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":96,"activation":"relu","recurrent_activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"unit_forget_bias":true,"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.5,"recurrent_dropout":0.0,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,96]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_9","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":32,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,96]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_5","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_10","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,32]}}],"build_input_shape":[null,1,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"GRU":{"config":{"name":"sequential_1","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,1,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_1","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"GRU","config":{"name":"gru","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":false,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":64,"activation":"relu","recurrent_activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.1,"recurrent_dropout":0.0,"reset_after":true,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,48]}},{"module":"keras.layers","class_name":"Flatten","config":{"name":"flatten","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"data_format":"channels_last"},"registered_name":null,"build_config":{"input_shape":[null,64]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_3","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":32,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.01},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,64]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.30000000000000004,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_4","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,32]}}],"build_input_shape":[null,1,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.0010000000474974513,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}},"RNN":{"config":{"name":"sequential_3","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"layers":[{"module":"keras.layers","class_name":"InputLayer","config":{"batch_shape":[null,1,48],"dtype":"float32","sparse":false,"ragged":false,"name":"input_layer_3","optional":false},"registered_name":null},{"module":"keras.layers","class_name":"GRU","config":{"name":"gru_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"return_sequences":false,"return_state":false,"go_backwards":false,"stateful":false,"unroll":false,"zero_output_for_mask":false,"units":16,"activation":"relu","recurrent_activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"recurrent_initializer":{"module":"keras.initializers","class_name":"Orthogonal","config":{"seed":null,"gain":1.0},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"recurrent_regularizer":null,"bias_regularizer":null,"activity_regularizer":null,"kernel_constraint":null,"recurrent_constraint":null,"bias_constraint":null,"dropout":0.1,"recurrent_dropout":0.0,"reset_after":true,"seed":null},"registered_name":null,"build_config":{"input_shape":[null,1,48]}},{"module":"keras.layers","class_name":"Flatten","config":{"name":"flatten_2","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"data_format":"channels_last"},"registered_name":null,"build_config":{"input_shape":[null,16]}},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_7","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":320,"activation":"relu","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":{"module":"keras.regularizers","class_name":"L2","config":{"l2":0.0001},"registered_name":null},"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,16]}},{"module":"keras.layers","class_name":"Dropout","config":{"name":"dropout_4","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"rate":0.2,"seed":null,"noise_shape":null},"registered_name":null},{"module":"keras.layers","class_name":"Dense","config":{"name":"dense_8","trainable":true,"dtype":{"module":"keras","class_name":"DTypePolicy","config":{"name":"float32"},"registered_name":null},"units":1,"activation":"sigmoid","use_bias":true,"kernel_initializer":{"module":"keras.initializers","class_name":"GlorotUniform","config":{"seed":null},"registered_name":null},"bias_initializer":{"module":"keras.initializers","class_name":"Zeros","config":{},"registered_name":null},"kernel_regularizer":null,"bias_regularizer":null,"kernel_constraint":null,"bias_constraint":null,"quantization_config":null},"registered_name":null,"build_config":{"input_shape":[null,320]}}],"build_input_shape":[null,1,48]},"compile_config":{"optimizer":{"module":"keras.optimizers","class_name":"RMSprop","config":{"name":"rmsprop","learning_rate":0.009999999776482582,"weight_decay":null,"clipnorm":null,"global_clipnorm":null,"clipvalue":null,"use_ema":false,"ema_momentum":0.99,"ema_overwrite_frequency":null,"loss_scale_factor":null,"gradient_accumulation_steps":null,"rho":0.9,"momentum":0.0,"epsilon":1e-07,"centered":false},"registered_name":null},"loss":"binary_crossentropy","loss_weights":null,"metrics":["accuracy"],"weighted_metrics":null,"run_eagerly":false,"steps_per_execution":1,"jit_compile":true}}}')

In [ ]:
if any(DL_JOBS.values()):
    import tensorflow as tf
    print('TensorFlow:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
    def build_deep_model(name, width):
        spec = MODEL_CONFIGS[name] if MODEL_JSON_DIR is None else json.loads(
            (Path(MODEL_JSON_DIR) / MODEL_FILES[name]).read_text())
        layers = spec['config']['layers']
        input_config = layers[0]['config']
        shape = tuple((input_config.get('batch_shape') or input_config['batch_input_shape'])[1:])
        if len(shape) == 1:
            input_shape = (width,)
        elif shape[0] == 1:
            input_shape = (1,width)
        elif shape[-1] == 1:
            input_shape = (width,1)
        else:
            raise ValueError(f'Unsupported JSON input shape: {shape}')
        model = tf.keras.Sequential([tf.keras.Input(shape=input_shape, dtype='float32')], name=name)
        for original in layers[1:]:
            layer = copy.deepcopy(original)
            layer.pop('build_config', None)
            model.add(tf.keras.layers.deserialize(layer))
        if model.output_shape != (None,1) or getattr(model.layers[-1].activation,'__name__','') != 'sigmoid':
            raise ValueError(f'{name}: expected one sigmoid output per peptide.')
        if name == 'RNN_JSON_GRU' and not any(isinstance(layer, tf.keras.layers.GRU) for layer in model.layers):
            raise ValueError('Register a corrected RNN JSON under its own model name before using it.')
        optimizer = tf.keras.optimizers.deserialize(spec['compile_config']['optimizer'])
        model.compile(optimizer=optimizer, loss=spec['compile_config']['loss'],
                      metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])
        return model, input_shape


In [ ]:
if any(DL_JOBS.values()):
    if any(DL_JOBS.values()):
        # TensorFlow is required only when a deep-learning model is selected.
        pass
    
    class SelectiveProgbarLogger(tf.keras.callbacks.ProgbarLogger):
        def __init__(self, verbose, epoch_interval, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.default_verbose = verbose
            self.epoch_interval = epoch_interval
        
        def on_epoch_begin(self, epoch, *args, **kwargs):
            self.verbose = (
                0 
                    if epoch % self.epoch_interval != 0 
                    else self.default_verbose
            )
            super().on_epoch_begin(epoch, *args, **kwargs)
            
    def fit_deep(name,feature,train,outer_train_ids,test_ids,fold,seed):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        fit_ids,val_ids=train_test_split(outer_train_ids,test_size=VAL_FRACTION,
            stratify=train.y[outer_train_ids],random_state=seed)
        assert not np.intersect1d(test_ids,np.concatenate([fit_ids,val_ids])).size
        scaler=StandardScaler().fit(train.X.iloc[fit_ids])
        model,shape=build_deep_model(name,train.X.shape[1])
        def transform(frame):
            return scaler.transform(frame).astype('float32',copy=False).reshape((-1,*shape))
        class_weights=compute_class_weight('balanced',classes=np.array([0,1]),y=train.y[fit_ids])
        prefix=RUN_DIR/f'{name}_{feature}_fold{fold}'
        roles=np.full(len(train.y),'outer_test',dtype=object); roles[fit_ids]='fit'; roles[val_ids]='inner_validation'
        pd.DataFrame(dict(row_index=np.arange(len(train.y)),role=roles)).to_csv(f'{prefix}_partition.csv',index=False)
        callbacks=[SelectiveProgbarLogger(verbose=0,epoch_interval=10),
            tf.keras.callbacks.EarlyStopping(monitor=MONITOR,patience=PATIENCE,mode=MODE,verbose=1,restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor=MONITOR,mode=MODE,factor=0.5,patience=10,min_lr=1e-6,verbose=1)]
        if SAVE_MODELS:
            callbacks.append(tf.keras.callbacks.ModelCheckpoint(filepath=f'{prefix}.keras',monitor=MONITOR,mode=MODE,save_best_only=True,verbose=0))
        history=model.fit(transform(train.X.iloc[fit_ids]),train.y[fit_ids],
            validation_data=(transform(train.X.iloc[val_ids]),train.y[val_ids]),
            batch_size=BATCH_SIZE,epochs=MAX_EPOCHS,shuffle=True,class_weight=dict(enumerate(class_weights.tolist())),
            callbacks=callbacks,verbose=0)
        # Explicitly load the validation-selected checkpoint to ensure evaluated and saved weights match.
        if SAVE_MODELS: model=tf.keras.models.load_model(f'{prefix}.keras')
        pd.DataFrame(history.history).rename_axis('epoch_zero_based').to_csv(f'{prefix}_history.csv')
        if SAVE_MODELS: joblib.dump(scaler,f'{prefix}_scaler.joblib')
        Path(f'{prefix}_settings.json').write_text(json.dumps(dict(input_shape=shape,scaler='StandardScaler',seed=seed,
            fold=fold,n_outer_train=len(outer_train_ids),n_fit=len(fit_ids),n_inner_val=len(val_ids),n_outer_test=len(test_ids),
            validation_fraction=VAL_FRACTION,monitor=MONITOR,patience=PATIENCE,max_epochs=MAX_EPOCHS,
            batch_size=BATCH_SIZE,class_weights=class_weights.tolist(),architecture=json.loads(model.to_json())),indent=2))
        p=model.predict(transform(train.X.iloc[test_ids]),batch_size=256,verbose=0).reshape(-1)
        return p,len(fit_ids),len(val_ids)
    
    dl_rows=[]
    for feature,models in DL_JOBS.items():
        if not models: continue
        train=load_feature(feature)['train_ibce']; splits=folds_for(train)
        for model in models:
            oof=np.full(len(train.y),np.nan)
            for fold,(train_ids,test_ids) in enumerate(splits,1):
                print(f'{feature}/{model}: fold {fold}/{K}',flush=True)
                started=time.perf_counter(); seed=MODEL_SEED+fold-1
                p,n_fit,n_val=fit_deep(model,feature,train,train_ids,test_ids,fold,seed)
                oof[test_ids]=p
                row=evaluate_fold(train,test_ids,p,model,feature,fold,seed)
                row.update(n_outer_train=len(train_ids),n_fit=n_fit,n_inner_val=n_val,seconds=time.perf_counter()-started)
                dl_rows.append(row); ALL_ROWS.append(row)
                save_metrics(dl_rows,'deep_learning'); save_metrics(ALL_ROWS,'all_models')
            save_oof(train,oof,model,feature)


In [ ]:
# Final fold-wise results and mean ± sample SD for every selected metric.
if ALL_ROWS:
    print('\nFINAL FOLD-WISE METRICS')
    print(pd.DataFrame(ALL_ROWS)[['feature_set','model','fold',*METRICS]].round(4).to_string(index=False))
    final_summary=save_metrics(ALL_ROWS,'all_models')
    try:
        from IPython.display import display
        display(final_summary)
    except ImportError:
        pass
    print('\nOutputs:',RUN_DIR.resolve())
else:
    print('No completed folds. Run the selected model sections first.')
